# SVD for Recommender Systems

## Learning Objectives

1. **Explain** how SVD decomposes the utility matrix into latent factors
2. **Handle** missing entries via mean imputation before SVD
3. **Generate** recommendations from the low-rank reconstruction
4. **Describe** why regularized SGD variants outperform plain SVD

## SVD Applied to the Utility Matrix

Given a partially observed utility matrix $M$ (rows = users, columns = items):

**Step 1: Fill missing entries** (e.g., with row/column means)

**Step 2: Compute SVD:** $M_{\text{filled}} = U \Sigma V^\top$

**Step 3: Truncate to rank $k$:** $\hat{M} = U_k \Sigma_k V_k^\top$

**Step 4: Use $\hat{M}_{ui}$ as predicted rating** for any missing entry

**Interpretation:**
- $V^\top$ rows = item "concept" vectors
- $U$ rows = user embeddings in concept space
- $\Sigma$ weights each concept dimension by its importance

In [1]:
import math, random

def dot(a, b): return sum(x*y for x,y in zip(a,b))
def norm(v): return math.sqrt(sum(x**2 for x in v))
def normalize(v): n=norm(v); return [x/n for x in v] if n>0 else v
def mat_vec(A, v): return [sum(A[i][j]*v[j] for j in range(len(v))) for i in range(len(A))]
def transpose(A): return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def fill_missing(M, fill='row_mean'):
    """Fill missing (None) entries."""
    m, n = len(M), len(M[0])
    M_filled = [row[:] for row in M]
    for i in range(m):
        known = [M[i][j] for j in range(n) if M[i][j] is not None]
        mean = sum(known)/len(known) if known else 3.0
        for j in range(n):
            if M_filled[i][j] is None:
                M_filled[i][j] = mean
    return M_filled

def svd_top1(A, n_iter=300, seed=0):
    "Single SVD component via power iteration."
    m, n = len(A), len(A[0])
    rng = random.Random(seed)
    v = normalize([rng.gauss(0,1) for _ in range(n)])
    At = transpose(A)
    for _ in range(n_iter):
        u = normalize(mat_vec(A, v))
        v = normalize(mat_vec(At, u))
    sigma = sum(u[i]*sum(A[i][j]*v[j] for j in range(n)) for i in range(m))
    return sigma, u, v

def svd_top_k(A, k, seed=0):
    "Top-k SVD via deflation."
    R = [row[:] for row in A]
    components = []
    for s in range(k):
        sigma, u, v = svd_top1(R, seed=s)
        components.append((sigma, u, v))
        for i in range(len(R)):
            for j in range(len(R[0])):
                R[i][j] -= sigma * u[i] * v[j]
    return components

def reconstruct(components, m, n):
    """Reconstruct rank-k approximation."""
    ""
    M = [[0.0]*n for _ in range(m)]
    for sigma, u, v in components:
        for i in range(m):
            for j in range(n):
                M[i][j] += sigma * u[i] * v[j]
    return M

# Small utility matrix (None = unobserved)
users = ['Alice','Bob','Carol','Dave','Eve']
items = ['M1','M2','M3','M4','M5','M6']
M = [
    [5,   3,   None, 1,   None, 2  ],
    [4,   None,4,    1,   None, 2  ],
    [None,1,   None, 5,   4,   None],
    [None,None,1,    4,   5,   None],
    [1,   None,4,   None, 4,   1  ],
]
print("Utility matrix (None = unobserved):")
print(f"{'':>7}", end='')
for item in items: print(f"{item:>6}", end='')
print()
for u, row in zip(users, M):
    print(f"{u:>7}", end='')
    for v in row:
        print(f"{'?':>6}" if v is None else f"{v:>6}", end='')
    print()

Utility matrix (None = unobserved):
           M1    M2    M3    M4    M5    M6
  Alice     5     3     ?     1     ?     2
    Bob     4     ?     4     1     ?     2
  Carol     ?     1     ?     5     4     ?
   Dave     ?     ?     1     4     5     ?
    Eve     1     ?     4     ?     4     1


In [2]:
# Fill and apply SVD
M_filled = fill_missing(M, fill='row_mean')

# Rank-2 SVD
components = svd_top_k(M_filled, k=2)
sigmas = [c[0] for c in components]
print(f"""
Singular values: σ₁={sigmas[0]:.3f}, σ₂={sigmas[1]:.3f}""")
print(f"Energy captured: {(sigmas[0]**2+sigmas[1]**2)/sum(s**2 for s,_,_ in svd_top_k(M_filled,4)):.2%}")

M_hat = reconstruct(components, len(M), len(M[0]))

# Show predicted ratings for missing entries
print("""
Predicted ratings for unobserved entries:""")
print(f"""{'User':>8} {'Item':>6} {'Predicted':>10}""")
for i, user in enumerate(users):
    for j, item in enumerate(items):
        if M[i][j] is None:
            pred = max(1, min(5, M_hat[i][j]))
            print(f"{user:>8} {item:>6} {pred:>10.2f}")


Singular values: σ₁=16.373, σ₂=4.321
Energy captured: 94.56%

Predicted ratings for unobserved entries:
    User   Item  Predicted
   Alice     M3       3.56
   Alice     M5       2.86
     Bob     M2       2.96
     Bob     M5       2.87
   Carol     M1       2.81
   Carol     M3       2.61
   Carol     M6       3.06
    Dave     M1       2.95
    Dave     M2       2.30
    Dave     M6       3.00
     Eve     M2       2.00
     Eve     M4       2.66


## Why Not Plain SVD?

**Problem 1:** Filling missing values with means introduces **bias** — the mean-filled entries are not observed and may distort the SVD.

**Problem 2:** No regularization → overfits to observed ratings.

**Better approaches:**
- **Regularized SVD / SGD-based MF:** only minimizes over observed entries (no imputation needed)
- **ALS (Alternating Least Squares):** solve for $U$ and $V$ in alternation; parallelizable
- **Netflix Prize solution:** Ensemble with SVD++, which adds implicit feedback

The "SVD" label in recommender systems usually refers to regularized matrix factorization (not exact SVD).